# Chạy Qwen / Llama trên Colab — thí nghiệm toàn bộ log

Thiết kế: `md/THIET_KE_TEST_TOAN_BO_LOG_25112026.md`. Chạy lần lượt từ trên xuống. Trước đó tải `colab_pack.zip` lên `/content` (bảng Files bên trái). Bị ngắt phiên thì chạy lại ô 1–6 rồi ô 7: script bỏ qua ca đã có trong CSV trên Drive.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Google Drive (bấm cho phép khi được hỏi) — CSV kết quả ghi vào đây để không mất khi ngắt phiên
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/DACN_colab'
os.makedirs(OUT_DIR, exist_ok=True)
print(OUT_DIR)

In [ ]:
# 3. Giải nén gói
!rm -rf /content/winlog-agent && unzip -q -o /content/colab_pack.zip -d /content
!ls /content/winlog-agent/eval_real/cases | wc -l

In [ ]:
# 4. Cài Ollama (script chính thức ollama.com) và bật server chạy nền
!apt-get -qq install -y zstd pciutils > /dev/null
OLLAMA_PIN = "0.40.18"  # cùng bản với lượt đã khoá (eval_real/prompt_freeze.json)
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_PIN} sh
import subprocess, time, requests
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://127.0.0.1:11434', timeout=2); break
    except requests.RequestException:
        time.sleep(1)
print(requests.get('http://127.0.0.1:11434').text)

In [ ]:
# 5. Tải model và KIỂM TRA trùng mã với máy nhóm (khác mã -> dừng, kết quả không so được)
EXPECTED = {'qwen2.5:3b-instruct-q4_K_M': '357c53fb659c', 'llama3.2:3b-instruct-q4_K_M': 'a80c4f17acd5'}
for m in EXPECTED:
    subprocess.run(['ollama', 'pull', m], check=True, stdout=subprocess.DEVNULL)
listing = subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout
print(listing)
ids = {line.split()[0]: line.split()[1] for line in listing.splitlines()[1:] if line.strip()}
bad = {m: ids.get(m) for m, i in EXPECTED.items() if ids.get(m) != i}
assert not bad, f'Mã model khác máy nhóm: {bad}'
OLLAMA_VERSION = subprocess.run(['ollama', '--version'], capture_output=True, text=True).stdout.strip()
assert OLLAMA_PIN in OLLAMA_VERSION, f'Ollama {OLLAMA_VERSION} khác bản đã khoá {OLLAMA_PIN}'
print('OK', OLLAMA_VERSION)

In [ ]:
# 6. Thư viện (Colab thường đã có sẵn)
!pip -q install 'pydantic>=2.8' 'requests>=2.31'

In [ ]:
# 7. CHẠY. Sửa 4 biến rồi chạy ô này. Mỗi lượt một TAG (một file CSV trên Drive).
SPLIT = 'dev'            # dev | test
TOOLS = 'qwen,llama'
TAG = 'full_dev1'        # full_dev1, full_dev2, full_test, full_base
NOISE_GUIDANCE = '1'     # '0' = lượt đối chứng (không có hướng dẫn nhiễu)
ONLY = ''                # vd 'eval_real/dev_round2.txt' cho vòng 2
out = f'{OUT_DIR}/combined_results_{TAG}.csv'
only_arg = f'--only {ONLY}' if ONLY else ''
!cd /content/winlog-agent && NOISE_GUIDANCE={NOISE_GUIDANCE} python eval_real/run_combined_eval.py --set full --split {SPLIT} --tools {TOOLS} --tag {TAG} --no-gpu-guard --out "{out}" {only_arg}

In [ ]:
# 8. Tình hình file kết quả
import csv, collections, glob
for p in sorted(glob.glob(f'{OUT_DIR}/combined_results_*.csv')):
    rows = [r for r in csv.DictReader(open(p, encoding='utf-8')) if r.get('pred')]
    print(p.split('/')[-1], len(rows), dict(collections.Counter((r['tool'], r['pred']) for r in rows)))

In [ ]:
# 9. Thông tin khoá prompt (chép vào eval_real/prompt_freeze.json trên máy)
import hashlib, sys, json, datetime, os
os.environ['ENABLE_EXTENDED_IDS'] = '1'  # prompt của lượt --set full (hệ thống thật mặc định tắt nhóm ID mới)
sys.path.insert(0, '/content/winlog-agent/core')
import analyze as A
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(json.dumps({'frozen_at': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'analyze_sha256': hashlib.sha256(open('/content/winlog-agent/core/analyze.py', 'rb').read()).hexdigest(),
    'prompt_fingerprint': A.prompt_fingerprint(), 'ollama_colab': OLLAMA_VERSION, 'gpu': gpu}, indent=1))